[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **Smart AI Waste Classifier**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

## Project Overview  
This project leverages a **custom-trained YOLOv11n-seg model** to perform high-speed instance segmentation for automated waste management.  
Using a manually annotated dataset of four waste categories, the system generates **pixel-level masks** to accurately identify object boundaries and surface area.

Unlike standard classification models, this approach provides **spatial intelligence**, enabling robotic sorting and precise volume estimation in real-world environments.

To ensure high performance and clarity, the system incorporates:

- **Supervised Instance Segmentation**  
  Trained on a custom dataset labeled using Labellerr, capturing real-world variations in waste.

- **Retina Mask Technology**  
  Generates masks at native resolution to preserve sharp edges on thin objects like plastic bags and bottle necks.

- **Optimized Mobile Backbone**  
  Built on the YOLOv11 Nano architecture, balancing speed and accuracy for edge deployment.

- **Dense Scene Processing**  
  Accurately distinguishes overlapping or touching objects, such as mixed waste items in cluttered environments.

---

## Real-World Applications  

### Robotic Sorting Pipelines  
Deployed in Material Recovery Facilities (MRFs) to guide robotic arms for precise waste sorting on conveyor belts.

### Automated Waste Auditing  
Tracks waste type and volume across operational shifts, enabling data-driven sustainability reporting.

### Contamination Detection  
Detects incorrect materials in recycling streams (e.g., plastic in paper bins), preventing batch rejection.

### Smart Collection Analytics  
Powers intelligent bins that analyze their contents and notify collection systems when capacity thresholds are reached.

---

## Key Features  

- **4-Class Custom Architecture**  
  Trained to detect bottles, plastic containers, plastic bags, and crumpled paper with high accuracy (mAP).

- **Labellerr-Verified Annotations**  
  Built on high-quality labeled data, ensuring strong performance and minimal background noise interference.

- **Clean Mask Visualization**  
  Outputs only high-contrast segmentation masks, removing bounding boxes and confidence scores for a clean interface.

- **Edge-Ready Performance**  
  Optimized for real-time inference on lightweight hardware such as smart cameras and embedded systems.

---

## Annotate your Custom dataset using Labellerr

 ***1. Visit the [Labellerr](https://www.labellerr.com/?utm_source=githubY&utm_medium=social&utm_campaign=github_clicks) website and click **“Sign Up”**.*** 

 ***2. After signing in, create your workspace by entering a unique name.***

 ***3. Navigate to your workspace’s API keys page (e.g., `https://<your-workspace>.labellerr.com/workspace/api-keys`) to generate your **API Key** and **API Secret**.***

 ***4. Store the credentials securely, and then use them to initialise the SDK or API client with `api_key`, `api_secret`.*** 



## Import Libraries

This section imports all the required libraries used throughout the project for computer vision, visualization, deep learning, and structured coding.


In [1]:
!pip install ultralytics opencv-python matplotlib cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for cv2


In [2]:
!git clone https://github.com/Labellerr/yolo_finetune_utils.git

Cloning into 'yolo_finetune_utils'...


## 🎞️ Random Frame Extraction from Video

Extracts a fixed number of high-quality frames from one or more videos to create an image dataset for annotation and training.

### 🔹 Purpose
- Convert raw manufacturing videos into individual image frames  
- Perform random sampling to avoid frame bias  
- Prepare data for annotation and YOLO training  


In [ ]:
from yolo_finetune_utils.frame_extractor import extract_random_frames

extract_random_frames(
    paths=['download (1).mp4'],
    total_images=50,
    out_dir="dataset_Frames",
    jpg_quality=100,
    seed=42
)

[✓] Extracted 15 frames to folder: dataset_Frames


## 📥 Download Annotations from Labellerr

After completing data labeling on the **Labellerr** platform, export the annotations in **COCO JSON format**.

Download the COCO JSON file from the Labellerr website and upload it into this project workspace to use it for further dataset preparation and training.

This COCO JSON file will be used in the next steps for:
- Frame–annotation alignment
- COCO → YOLO format conversion
- Model training and evaluation


# COCO to YOLO Format Conversion

Converts COCO-style segmentation annotations to YOLO segmentation dataset format.  
- Requires: `annotation.json` and images in `frames_output` directory.
- Output: Generated YOLO dataset folder.
- Parameters: allows train/val split, shuffling, and verbose mode.


In [3]:
from yolo_finetune_utils.coco_yolo_converter.seg_converter import coco_to_yolo_converter

coco_to_yolo_converter(
    json_path="c372b4a1-b7cb-4105-a32d-ab24d2c8d948.json",
    images_dir="dataset_Frames",
    output_dir="yolo_Dataset",
    use_split=True,
    train_ratio=0.9,
    val_ratio=0.1,
    test_ratio=0,
    shuffle=True,
    verbose=True
)

Conversion complete. Stats: {'train': 13, 'val': 1, 'test': 1}


{'stats': {'train': 13, 'val': 1, 'test': 1}, 'output_dir': 'yolo_Dataset'}

# Load and Train YOLO Segmentation Model

Loads the YOLO segmentation model and trains it using the converted YOLO dataset.
- Data: Path to YOLO-style `data.yaml`
- Parameters: epochs, image size, batch size, device, dataloader workers, experiment name.


In [ ]:
# Initialize the Extra Large model
model = YOLO('yolo11n-seg.pt') 

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    project='/kaggle/working/pi_components',
    name='train_run_4',
    device=0,
    
    # --- AUGMENTATION HAPPENS HERE ---
    degrees=15.0,    # Randomly rotates images up to +/- 15 degrees
    fliplr=0.5,      # 50% chance to flip left-right
    flipud=0.5       # 50% chance to flip upside-down
)

# Inference Pipeline Overview

This script serves as the **inference pipeline** for the waste classification project.  
It automates the process of identifying and segmenting different types of waste using a custom-trained YOLOv11n-seg model.

---

## Key Functional Steps  

### Model Loading  
Initializes the **YOLOv11n-seg** architecture using the custom weights file (`best.pt`).  
These weights are trained on the Labellerr dataset and enable accurate segmentation of four waste categories:
- Bottles  
- Containers  
- Plastic Bags  
- Crumpled Paper  

---

### High-Resolution Masking  
Enables `retina_masks=True` to generate segmentation masks at the original video resolution.  
This ensures sharp, pixel-accurate boundaries, which is especially important for thin materials like plastic bags and detailed textures like crumpled paper.

---

### Visual Optimization  
Configures a **Clean UI** output by:
- Displaying color-coded masks for each waste category  
- Hiding bounding boxes and confidence scores  

This keeps the visualization focused on material separation while maintaining clarity.

---

### Memory-Efficient Streaming  
Uses `stream=True` to process the video frame-by-frame.  
This reduces GPU memory usage and prevents Out-of-Memory (OOM) errors during long inference runs on Kaggle’s T4 environment.

---

### Automated Export  
The `save=True` flag enables automatic result saving.  
Processed frames are compiled into a high-definition video file and stored in:

`/kaggle/working/runs/segment/predict`

This allows easy access for download and further analysis.

---

In [ ]:
from ultralytics import YOLO

# 1. Load your newly trained custom model
# Make sure "best.pt" is in the same directory, or provide the full path
model = YOLO("/kaggle/working/pi_components/train_run_4/weights/best.pt") 

video_path = "/kaggle/input/datasets/aaryanaggarwal5040/wastefilevid/download (1).mp4" # Update this to your actual video file name

print(f"Starting inference on {video_path}...")
results = model.predict(
    source=video_path,
    save=True, 
    conf=0.2,
    show_boxes=True,    # Keep this False to hide the rectangles
    show_labels=True,    # Keep this True to see the names
    show_conf=False,
    retina_masks=True,
    stream=True,
    # --- ADD THIS TO SHRINK THE TEXT ---
    line_width=2         # This will make the font significantly smaller and thinner
)

# 3. Execute the video processing loop
# Because stream=True creates a generator, we must iterate through it to process the video
for frame_result in results:
    pass 

print("\nInference complete!")
print("Look for your newly generated video inside the 'runs/segment/predict/' folder.")

---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
